In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "wslr": MTU.get_fully_qualified_table_name("oracle", "bud", "wslr"),
    "bud_user": MTU.get_fully_qualified_table_name("oracle", "bud", "budnet_user"),
    "bud_wslr": MTU.get_fully_qualified_table_name("oracle", "bud", "budnet_wslr"),
    "vw_wslr": MTU.get_fully_qualified_table_name("oracle", "crd", "vw_wslr_base_dom"),
    "vw_rrst": MTU.get_fully_qualified_table_name("oracle", "crd", "vw_rrst_raw_base"),
    # gold source table
    "wslr_addr": MTU.get_fully_qualified_table_name("commercial", "wholesaler", "wholesaler_address"),
}

In [0]:
# business transformations here
mktng_st_cd = (
    F.when(
        (F.col("wslr.wslr_state_code").isNull()) | (F.col("wslr.wslr_state_code") == "99"),
        F.col("wslr.wslr_state_code"),
    )
    .when(F.col("wslr.wslr_state_code").isin("CB", "PR"), "CR")
    .when(F.col("wslr.region_code") == "17", "ML")
    .when((F.col("wslr.wslr_state_code") < "AK") | (F.col("wslr.region_code") == "90"), "EX")
    .otherwise(F.col("wslr.wslr_state_code"))
)

sls_dirctr_nm = F.when(
    F.expr("sd_rrst.last_nm ||', '|| sd_rrst.first_nm == vp_rrst.last_nm ||', '|| vp_rrst.first_nm"), None
).otherwise(F.expr("sd_rrst.last_nm ||', '|| sd_rrst.first_nm"))

rtl_sls_mgr_nm = (
    F.when(F.expr("rsm_rrst.last_nm ||', '|| rsm_rrst.first_nm == vp_rrst.last_nm ||', '|| vp_rrst.first_nm"), None)
    .when(
        F.concat(F.col("rsm_rrst.last_nm"), F.lit(", "), F.col("rsm_rrst.first_nm")).isNull(),
        F.concat(F.col("sd_rrst.last_nm"), F.lit(", "), F.col("sd_rrst.first_nm")),
    )
    .otherwise(F.expr("rsm_rrst.last_nm ||', '|| rsm_rrst.first_nm"))
)

In [0]:
# mapping from silver tables to gold wholesaler table
wslr_col_mappings = {
    "wslr_nbr": F.col("wslr.wslr_number"),
    "wslr_typ_cd": F.col("wslr.account_type"),
    "wslr_nm": F.col("wslr.wslr_name"),
    "wslr_st_cd": F.col("wslr.wslr_state_code"),
    "mktng_st_cd": mktng_st_cd,
    "tel_nbr": F.col("vw_wslr.phone_nbr"),
    "fax_nbr": F.col("vw_wslr.fax_nbr"),
    "budnet_actv_flg": F.when(F.col("wslr.account_type") == "BR", None).otherwise(F.col("bud_wslr.active_code")),
    "srce_brwy_cd": F.col("wslr.source_brewery_cd"),
    "fob_brwy_cd": F.col("wslr.fob_brewery_code"),
    "invty_plan_brwy_cd": F.col("wslr.ip_brewery_cd"),
    "wslr_invty_coord_cd": F.col("wslr.ip_coordinatr_cd"),
    "mh_wslr_nbr": F.col("wslr.mrkting_acct_nbr"),
    "serv_cntr_brwy_cd": F.col("wslr.sup_center_cde"),
    "pri_trnsprt_mode_cd": F.col("wslr.shipping_mode_code"),
    "east_regn_rr_flg": F.col("wslr.east_rgn_railrd_cd"),
    "ab_prod_excl_flg": F.col("wslr.excl_ab_wslr_flg"),
    "ppd_shpmt_cd": F.col("wslr.prepd_inpt_shp_flg"),
    "mil_brkout_cd": F.col("wslr.military_brkout_cd"),
    "mca_cd": F.col("wslr.media_coverage_cd"),
    "alpha_admin_cd": F.col("wslr.alpha_mrkting_nbr"),
    "sls_regn_cd": F.col("wslr.region_code"),
    "sls_dirctr_cd": F.col("wslr.sales_area_code"),
    "sls_dirctr_nm": sls_dirctr_nm,
    "rtl_sls_mgr_cd": F.col("wslr.retail_sales_code"),
    "rtl_sls_mgr_nm": rtl_sls_mgr_nm,
    "regn_vice_pres_nm": F.concat(F.col("vp_rrst.last_nm"), F.lit(", "), F.col("vp_rrst.first_nm")),
    "sls_start_tsp": F.col("bud_wslr.sales_start_date").cast(TimestampType()),
    "dqst_cd": F.col("bud_user.verf_group_text"),
    "shp_to_wslr_nm": F.col("ship_wslr_addr.ship_wslr_nm"),
    "shp_to_wslr_addr_1_txt": F.col("ship_wslr_addr.ship_wslr_addr_1_txt"),
    "shp_to_wslr_addr_2_txt": F.col("ship_wslr_addr.ship_wslr_addr_2_txt"),
    "shp_to_wslr_addr_3_txt": F.col("ship_wslr_addr.ship_wslr_addr_3_txt"),
    "shp_to_wslr_city_nm": F.col("ship_wslr_addr.ship_wslr_city_nm"),
    "shp_to_wslr_st_cd": F.col("ship_wslr_addr.ship_wslr_st_cd"),
    "shp_to_wslr_postal_cd": F.col("ship_wslr_addr.ship_wslr_postal_cd"),
    "mail_to_wslr_nm": F.col("mail_wslr_addr.mail_wslr_nm"),
    "mail_to_wslr_addr_1_txt": F.col("mail_wslr_addr.mail_wslr_addr_1_txt"),
    "mail_to_wslr_addr_2_txt": F.col("mail_wslr_addr.mail_wslr_addr_2_txt"),
    "mail_to_wslr_addr_3_txt": F.col("mail_wslr_addr.mail_wslr_addr_3_txt"),
    "mail_to_wslr_city_nm": F.col("mail_wslr_addr.mail_wslr_city_nm"),
    "mail_to_wslr_st_cd": F.col("mail_wslr_addr.mail_wslr_st_cd"),
    "mail_to_wslr_postal_cd": F.col("mail_wslr_addr.mail_wslr_postal_cd"),
    "bill_to_wslr_nm": F.col("bill_wslr_addr.bill_wslr_nm"),
    "bill_to_wslr_addr_1_txt": F.col("bill_wslr_addr.bill_wslr_addr_1_txt"),
    "bill_to_wslr_addr_2_txt": F.col("bill_wslr_addr.bill_wslr_addr_2_txt"),
    "bill_to_wslr_addr_3_txt": F.col("bill_wslr_addr.bill_wslr_addr_3_txt"),
    "bill_to_wslr_city_nm": F.col("bill_wslr_addr.bill_wslr_city_nm"),
    "bill_to_wslr_st_cd": F.col("bill_wslr_addr.bill_wslr_st_cd"),
    "bill_to_wslr_postal_cd": F.col("bill_wslr_addr.bill_wslr_postal_cd"),
}

In [0]:
# get recent scd Type 2 data from silver region roster table

# read silver table into DataFrame
vw_rrst_raw_df = spark.read.table(SOURCE_TABLES["vw_rrst"]).alias("vw_rrst")

# define window specification
window_spec = Window.partitionBy("sls_cd").orderBy(F.col("__created_tsp").desc(), F.col("eff_tsp").desc_nulls_last())

# define vp dataframe for vice president data
vp = (
    vw_rrst_raw_df.filter((F.col("posit_cd") == "005") & (F.col("typ_cd") == "RG") & (F.col("actv_flg") == "Y"))
    .withColumn("last_nm", F.trim(F.col("last_nm")))
    .withColumn("first_nm", F.trim(F.col("first_nm")))
    .withColumn("rn", F.row_number().over(window_spec))
)
# define sd dataframe for sales director data
sd = (
    vw_rrst_raw_df.filter((F.col("posit_cd") == "107") & (F.col("typ_cd") == "RG") & (F.col("actv_flg") == "Y"))
    .withColumn("last_nm", F.trim(F.col("last_nm")))
    .withColumn("first_nm", F.trim(F.col("first_nm")))
    .withColumn("rn", F.row_number().over(window_spec))
)
# define rsm dataframe for regional sales manager data
rsm = (
    vw_rrst_raw_df.filter(
        (F.col("posit_cd").isin("108", "109", "110")) & (F.col("typ_cd") == "RG") & (F.col("actv_flg") == "Y")
    )
    .withColumn("last_nm", F.trim(F.col("last_nm")))
    .withColumn("first_nm", F.trim(F.col("first_nm")))
    .withColumn("rn", F.row_number().over(window_spec))
)
rrst_cols = ["typ_cd", "posit_cd", "eff_tsp", "actv_flg", "sls_cd", "last_nm", "first_nm", "__created_tsp", "__source"]
# select desired columns and union the dataframes
rrst_filtered_df = (
    vp.select(*rrst_cols)
    .where(F.col("rn") == 1)
    .unionAll(sd.select(*rrst_cols).where(F.col("rn") == 1))
    .unionAll(rsm.select(*rrst_cols).where(F.col("rn") == 1))
)

In [0]:
try:
    # Base DF for joins
    left_df = spark.read.table(SOURCE_TABLES["wslr"]).alias("wslr")

    # other tables to join to base
    vw_wslr_df = (
        spark.read.table(SOURCE_TABLES["vw_wslr"])
        .select("wslr_number", "phone_nbr", "fax_nbr", "start_dt", "mail_cd_ltr_flg")
        .alias("vw_wslr")
    )
    bud_wslr_df = (
        spark.read.table(SOURCE_TABLES["bud_wslr"])
        .select("wslr_number", "active_code", "sales_start_date")
        .alias("bud_wslr")
    )
    bud_user_df = (
        spark.read.table(SOURCE_TABLES["bud_user"])
        .select("primary_budnet_nbr", "verf_group_text")
        .alias("cognos_sumry")
    )
    wslr_addr_df = spark.read.table(SOURCE_TABLES["wslr_addr"]).alias("wslr_addr")

    # join silver tables
    left_df = left_df.join(other=vw_wslr_df, on=F.col("wslr.wslr_number") == F.col("vw_wslr.wslr_number"), how="left")
    left_df = left_df.join(other=bud_wslr_df, on=F.col("wslr.wslr_number") == F.col("bud_wslr.wslr_number"), how="left")
    left_df = left_df.join(
        other=bud_user_df.alias("bud_user"),
        on=F.col("wslr.wslr_number") == F.col("bud_user.primary_budnet_nbr"),
        how="left",
    )

    # complex join on region roster silver table
    left_df = left_df.join(
        other=rrst_filtered_df.alias("vp_rrst"),
        on=F.expr("vp_rrst.sls_cd = concat(wslr.region_code, '00', '00')"),
        how="left",
    )
    left_df = left_df.join(
        other=rrst_filtered_df.alias("sd_rrst"),
        on=F.expr("sd_rrst.sls_cd = concat(wslr.region_code, wslr.sales_area_code, '00')"),
        how="left",
    )
    left_df = left_df.join(
        other=rrst_filtered_df.alias("rsm_rrst"),
        on=F.expr("rsm_rrst.sls_cd = concat(wslr.region_code, wslr.sales_area_code, wslr.retail_sales_code)"),
        how="left",
    )

    # join gold wholesaler_address table
    left_df = left_df.join(
        other=(
            wslr_addr_df.filter(F.col("wslr_addr.wslr_addr_typ_cd") == "SHIPTO")
            .select([F.col(col_name).alias("ship_" + col_name) for col_name in wslr_addr_df.columns])
            .alias("ship_wslr_addr")
        ),
        on=F.col("wslr.wslr_number") == F.col("ship_wslr_addr.ship_wslr_nbr"),
        how="left_outer",
    )
    left_df = left_df.join(
        other=(
            wslr_addr_df.filter(F.col("wslr_addr.wslr_addr_typ_cd") == "MAILTO")
            .select([F.col(col_name).alias("mail_" + col_name) for col_name in wslr_addr_df.columns])
            .alias("mail_wslr_addr")
        ),
        on=F.col("wslr.wslr_number") == F.col("mail_wslr_addr.mail_wslr_nbr"),
        how="left_outer",
    )
    left_df = left_df.join(
        other=(
            wslr_addr_df.filter(F.col("wslr_addr.wslr_addr_typ_cd") == "BILLTO")
            .select([F.col(col_name).alias("bill_" + col_name) for col_name in wslr_addr_df.columns])
            .alias("bill_wslr_addr")
        ),
        on=F.col("wslr.wslr_number") == F.col("bill_wslr_addr.bill_wslr_nbr"),
        how="left_outer",
    )

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(wslr_col_mappings)
        .select(*wslr_col_mappings.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
    )

    medallion.drop_duplicates()

except Exception as e:
    medallion.logger.error(f"Error processing wholesaler source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing wholesaler table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )